In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# 1. Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# 2. Load dataset (Make sure 'IMDB Dataset.csv' matches your uploaded filename)
df = pd.read_csv('IMDBDataset.csv')

# 3. Setup stop words and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# 4. Define the text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""

    # a. Convert to lowercase
    text = text.lower()

    # b. Remove HTML tags like <br />
    text = re.sub(r'<.*?>', '', text)

    # c. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # d. Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # e. Remove stop words and reduce words to root form
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(cleaned_words)

# 5. Apply cleaning function using 'review' column
print("Cleaning reviews... (this takes around 10–20 seconds for 50,000 rows)")
df['clean_text'] = df['review'].apply(clean_text)

# 6. Save the newly cleaned dataset
df.to_csv('cleaned_dataset.csv', index=False)

# 7. Display cleaned results
print("\n--- CLEANED DATA SUCCESS ---")
print(df[['review', 'clean_text']].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lee23\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lee23\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Cleaning reviews... (this takes around 10–20 seconds for 50,000 rows)

--- CLEANED DATA SUCCESS ---
                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                          clean_text  
0  one reviewer mentioned watching oz episode you...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically there family little boy jake think t...  
4  petter matteis love time money visually stunni...  


In [3]:
# 1. Install Hugging Face Transformers library
!pip install -q transformers torch

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import pipeline

# 2. Load the cleaned dataset
df = pd.read_csv('cleaned_dataset.csv').dropna(subset=['clean_text'])
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# 3. TF-IDF & Train-Test Split for ML Models
vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# Naïve Bayes
# ---------------------------------------------------------
print("Training Model (Naïve Bayes)...")
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

# ---------------------------------------------------------
# Support Vector Machine (SVM)
# ---------------------------------------------------------
print("Training Model (Linear SVM)...")
svm_model = LinearSVC(max_iter=2000)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

# ---------------------------------------------------------
# GPT / Transformer Model
# ---------------------------------------------------------
print("Loading Model (Transformer / Deep Learning Model)...")
# Using a lightweight Transformer pipeline for fast execution in Colab
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512,
)

# Testing Transformer model on 500 test samples (to save runtime in Colab)
test_reviews = df['review'].iloc[:500].tolist()
y_test_sub = df['label'].iloc[:500].values

print("Evaluating Transformer model on sample test set...")
transformer_preds = classifier(test_reviews)
y_pred_gpt = [
    1 if pred['label'] == 'POSITIVE' else 0 for pred in transformer_preds
]

# ---------------------------------------------------------
# EVALUATION & COMPARISON TABLE
# ---------------------------------------------------------
def get_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary'
    )
    return [round(acc, 4), round(prec, 4), round(rec, 4), round(f1, 4)]


results = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    '(Naïve Bayes)': get_metrics(y_test, y_pred_nb),
    '(SVM)': get_metrics(y_test, y_pred_svm),
    '(Transformer/GPT)': get_metrics(y_test_sub, y_pred_gpt),
}

results_df = pd.DataFrame(results)
print('\n================ FINAL RESULTS COMPARISON ================')
print(results_df)

# Save evaluation results to CSV for your report
results_df.to_csv('model_comparison_results.csv', index=False)

Training Model (Naïve Bayes)...
Training Model (Linear SVM)...
Loading Model (Transformer / Deep Learning Model)...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Evaluating Transformer model on sample test set...

================ FINAL RESULTS COMPARISON ================
      Metric  (Naïve Bayes)   (SVM)  (Transformer/GPT)
0   Accuracy         0.8547  0.8834             0.9040
1  Precision         0.8556  0.8801             0.9315
2     Recall         0.8561  0.8899             0.8608
3   F1-Score         0.8559  0.8849             0.8947
